# 🧭 systemone-lite: Phase 2 Spatial Intelligence Training (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fritzprix/systemone-lite/blob/main/notebooks/phase2_spatial_training_colab.ipynb)

> **Objective**: Train a unified **Spatial + Chess + General (v2)** System One model on Google Colab (T4 / L4 / A100 GPU).
>
> - **Model**: `Qwen/Qwen2.5-0.5B-Instruct`
> - **Data**: 50,000 mixed samples (20k 2D Spatial Games + 15k General Routing/Noul/Score + 15k Balanced Chess)
> - **Batch Size**: 8 (T4 16GB) or 16~32 (L4 24GB / A100)
> - **Target**: Export to Hugging Face Hub (`dwidlee/systemone-spatial-0.5b`)

## 1. Environment & GPU Verification
Verify GPU allocation. T4 (16GB), L4 (24GB), or A100 (40GB) are recommended.

In [ ]:
!nvidia-smi

## 2. Clone Repository & Install Dependencies

In [ ]:
# Clone systemone-lite
!git clone https://github.com/fritzprix/systemone-lite.git
%cd systemone-lite

# Install dependencies in editable mode
!pip install -q --upgrade pip
!pip install -q -e .
!pip install -q bitsandbytes huggingface_hub scipy python-chess

## 3. Build Unified Phase 2 Spatial Dataset (50,000 samples)

This step synthesizes 20,000 spatial game decisions across 4 distinct 2D environments:
- **Sokoban** (5,000): Box-pushing causal path planning
- **2048** (5,000): Strategic tile merging and spatial preservation
- **GridWorld** (5,000): BFS-distilled shortest path and obstacle avoidance
- **Connect Four** (5,000): Gravity-constrained 4-in-a-row tactical foresight

It then combines them with balanced General Routing and Chess datasets.

In [ ]:
!mkdir -p data

# 1. Generate 20,000 spatial samples (5,000 per game)
!python scripts/build_spatial_distill.py \
    --out data/spatial_train_20k.jsonl \
    --samples 5000 \
    --seed 42

# 2. Build 50k Unified Dataset (20k Spatial + 15k General + 15k Chess)
import json
import random

random.seed(42)
unified_samples = []

# Load 20k Spatial samples
with open('data/spatial_train_20k.jsonl', 'r') as f:
    spatial_rows = [json.loads(line) for line in f]
    unified_samples.extend(spatial_rows)
print(f"Loaded {len(spatial_rows)} spatial game samples.")

# Load General samples (if available in repo, take 15k)
if os.path.exists('data/train_50k.jsonl'):
    with open('data/train_50k.jsonl', 'r') as f:
        general_rows = [json.loads(line) for line in f]
        random.shuffle(general_rows)
        unified_samples.extend(general_rows[:15000])
        print(f"Added {len(general_rows[:15000])} general routing samples.")

# Load Chess samples (if available in repo, take 15k)
if os.path.exists('data/chess_train_100k.jsonl'):
    with open('data/chess_train_100k.jsonl', 'r') as f:
        chess_rows = [json.loads(line) for line in f]
        random.shuffle(chess_rows)
        unified_samples.extend(chess_rows[:15000])
        print(f"Added {len(chess_rows[:15000])} chess samples.")

random.shuffle(unified_samples)
target_path = 'data/mixed_spatial_train_50k.jsonl'
with open(target_path, 'w') as f:
    for row in unified_samples:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f"Successfully generated unified dataset: {target_path} ({len(unified_samples)} total rows)")

## 4. Run High-Throughput Phase 2 Training

On Google Colab T4 (16GB VRAM), we run with `--batch-size 8`.
On L4 / A100, you can increase `--batch-size 16` or `32`.
Training completes in ~30 to 45 minutes on T4 (or ~10 minutes on A100).

In [ ]:
!python scripts/chess_finetune.py \
    --data data/mixed_spatial_train_50k.jsonl \
    --out checkpoints/systemone-spatial-v2 \
    --model Qwen/Qwen2.5-0.5B-Instruct \
    --epochs 1 \
    --batch-size 8 \
    --lr 5e-5 \
    --tasks all \
    --stratified

## 5. Quick Verification & Demo
Verify that the newly trained Phase 2 model can solve spatial games.

In [ ]:
# Run quick non-interactive verification on 2048 & Sokoban
!python scripts/sokoban_demo.py --model checkpoints/systemone-spatial-v2 --moves 5
!python scripts/game2048_demo.py --model checkpoints/systemone-spatial-v2 --moves 5

## 6. Export Checkpoint to Hugging Face Hub

Upload the trained weights directly to Hugging Face so you can immediately load it anywhere.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, login

# Enter your Hugging Face write token (or store in Colab Secrets as 'HF_TOKEN')
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    hf_token = getpass.getpass("Enter your Hugging Face Write Token: ")

login(token=hf_token)

repo_id = "dwidlee/systemone-spatial-0.5b"  # Change to your target HF repo

api = HfApi()
api.create_repo(repo_id=repo_id, exist_ok=True, private=False)
api.upload_folder(
    folder_path="checkpoints/systemone-spatial-v2",
    repo_id=repo_id,
    commit_message="Upload Phase 2 Spatial-tuned System One (0.5B) checkpoint"
)
print(f"Successfully uploaded model checkpoint to: https://huggingface.co/{repo_id}")